# 13 · Lista de casa: Keras e TensorFlow

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

## Instruções

- Trabalhe em uma cópia deste notebook e substitua cada `...` pelo seu código.
- Use o Keras 3 com o motor TensorFlow, como em aula (`import keras`, e não `tensorflow.keras`).
- Cerca de metade da nota está nas respostas escritas.
- Antes de entregar, use **Reiniciar sessão e executar tudo**.

| Questão | Tema | Pontos |
|---|---|---|
| 1 | Do PyTorch para o Keras | 2,5 |
| 2 | *Callbacks* | 2,5 |
| 3 | `GradientTape` | 2,0 |
| 4 | Modernizar um código antigo | 2,0 |
| 5 | Que framework usar | 1,0 |

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gzip
import time
from pathlib import Path
from urllib.request import urlretrieve

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

SEMENTE = 42
assert keras.backend.backend() == "tensorflow", "Reinicie a sessão: o Keras já tinha sido importado com outro motor."

BASE = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/b2617bb6d3ffa2e429640350f613e3291e10b141/data/fashion/"
PASTA = Path("fashion_mnist")
PASTA.mkdir(exist_ok=True)
for arquivo in ["train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz"]:
    if not (PASTA / arquivo).exists():
        urlretrieve(BASE + arquivo, PASTA / arquivo)

with gzip.open(PASTA / "train-images-idx3-ubyte.gz") as f:
    imagens = np.frombuffer(f.read(), dtype=np.uint8, offset=16).reshape(-1, 28, 28)
with gzip.open(PASTA / "train-labels-idx1-ubyte.gz") as f:
    rotulos = np.frombuffer(f.read(), dtype=np.uint8, offset=8)

ordem = np.random.default_rng(SEMENTE).permutation(60_000)
ind_ajuste, ind_valid = ordem[:50_000], ordem[50_000:]
MEDIA, DESVIO = float(imagens[ind_ajuste].mean() / 255), float(imagens[ind_ajuste].std() / 255)


def preparar(lote_uint8):
    return (((lote_uint8.astype("float32") / 255) - MEDIA) / DESVIO).astype("float32")[..., np.newaxis]


x_ajuste, y_ajuste = preparar(imagens[ind_ajuste]), rotulos[ind_ajuste].astype("int64")
x_valid, y_valid = preparar(imagens[ind_valid]), rotulos[ind_valid].astype("int64")


def criar_rede():
    """A rede convolucional da aula, já compilada."""
    rede = keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        layers.Conv2D(16, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding="same", activation="relu"), layers.MaxPooling2D(2),
        layers.Flatten(), layers.Dense(64, activation="relu"), layers.Dense(10),
    ])
    rede.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    return rede


def parametros_treinaveis(modelo):
    return sum(int(np.prod(peso.shape)) for peso in modelo.trainable_weights)


print("keras", keras.__version__, "· tensorflow", tf.__version__, "· ajuste:", x_ajuste.shape, x_ajuste.dtype)

## Questão 1 (2,5 pontos): do PyTorch para o Keras

A rede abaixo, em PyTorch, é a variante com **normalização em lote** usada na aula 08. Traduza-a para Keras, no estilo que preferir, e guarde o modelo em `rede_q1`. Depois compile (como em `criar_rede`) e treine por 3 épocas, com lotes de 128 e validação, guardando o resultado do `fit` em `historico_q1`.

```python
class RedeComNormalizacao(nn.Module):                                   # PyTorch: imagens (N, 1, 28, 28)
    def __init__(self):
        super().__init__()
        self.extrator = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classificador = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 10))

    def forward(self, x):
        return self.classificador(self.extrator(x))
```

In [ ]:
# ✏️ Questão 1
keras.utils.set_random_seed(SEMENTE)
# SEU CÓDIGO AQUI
rede_q1 = ...
historico_q1 = ...

In [ ]:
# ✅ Conferência da questão 1 (não altere)
def _camadas(modelo):
    for camada in modelo.layers:
        yield camada
        if hasattr(camada, "layers"):
            yield from _camadas(camada)

assert sum(isinstance(c, layers.BatchNormalization) for c in _camadas(rede_q1)) == 2, "o modelo deve ter duas camadas de normalização em lote"
assert rede_q1.output_shape == (None, 10) and tuple(np.asarray(rede_q1(x_valid[:4], training=False)).shape) == (4, 10)
assert parametros_treinaveis(rede_q1) == 105_962, f"parâmetros treináveis: {parametros_treinaveis(rede_q1)}; o módulo em PyTorch tem 105.962"
assert rede_q1.count_params() == 105_962 + 96
assert len(historico_q1.history["val_accuracy"]) == 3 and historico_q1.history["val_accuracy"][-1] > 0.85
print(f"Formato correto: {parametros_treinaveis(rede_q1)} parâmetros treináveis, {rede_q1.count_params() - parametros_treinaveis(rede_q1)} não treináveis · acurácia de validação {historico_q1.history['val_accuracy'][-1]:.1%}")

**1b)** Liste as diferenças que você precisou tratar na tradução (forma da entrada, camadas, ativações, número de entradas das camadas). O Keras informa 96 parâmetros **não treináveis**, que não aparecem na contagem de parâmetros do PyTorch: o que são, e onde o PyTorch os guarda?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 2 (2,5 pontos): *callbacks*

a) Escreva a classe `Cronometro`, um *callback* próprio (herda de `keras.callbacks.Callback`) que guarda em `self.tempos` a duração, em segundos, de cada época. Use os métodos `on_train_begin(self, logs=None)`, `on_epoch_begin(self, epoch, logs=None)` e `on_epoch_end(self, epoch, logs=None)`.

b) Treine uma rede criada com `criar_rede()` por **até 12 épocas**, com lotes de 128 e validação, usando quatro *callbacks*: parada antecipada (`patience=2`, restaurando os melhores pesos), gravação do melhor modelo em `q2.keras`, registro em `q2.csv` (`keras.callbacks.CSVLogger`) e o seu `cronometro` (uma instância de `Cronometro`). Guarde o resultado do `fit` em `historico_q2`.

c) Leia `q2.csv` com o pandas e guarde em `registro_q2`.

In [ ]:
# ✏️ Questão 2
# SEU CÓDIGO AQUI
class Cronometro(keras.callbacks.Callback):
    ...


rede_q2 = criar_rede()
cronometro = Cronometro()
historico_q2 = ...
registro_q2 = ...

In [ ]:
# ✅ Conferência da questão 2 (não altere)
_epocas = len(historico_q2.history["val_loss"])
assert isinstance(cronometro, keras.callbacks.Callback) and len(cronometro.tempos) == _epocas and all(t > 0 for t in cronometro.tempos), "um tempo por época executada"
assert len(registro_q2) == _epocas and {"epoch", "loss", "val_loss", "val_accuracy"} <= set(registro_q2.columns)
assert Path("q2.keras").exists() and _epocas <= 12
_perda_final = rede_q2.evaluate(x_valid, y_valid, verbose=0)[0]
assert abs(_perda_final - min(historico_q2.history["val_loss"])) < 1e-3, "o modelo deveria ter terminado com os pesos da melhor época (restore_best_weights)"
_do_arquivo = keras.saving.load_model("q2.keras").evaluate(x_valid, y_valid, verbose=0)[0]
assert abs(_do_arquivo - _perda_final) < 1e-3, "o arquivo q2.keras deveria conter o melhor modelo"
print(f"Formato correto: {_epocas} épocas executadas; a perda de validação do modelo final ({_perda_final:.4f}) é a menor do histórico.")

**2d)** Quantas épocas foram executadas, e em qual estava o melhor modelo? Relacione cada um dos quatro *callbacks* com o trecho da função `ajustar` da aula 08 que fazia o mesmo trabalho. Cite uma vantagem e uma limitação de organizar o treino com *callbacks*, em comparação com o laço escrito à mão.

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 3 (2,0 pontos): `GradientTape`

Vamos refazer, em TensorFlow, a regressão linear da aula 07.

a) Com `tf.GradientTape`, calcule o gradiente da perda `média((X @ w − y)²)` em relação à variável `w_q3`, e guarde em `grad_tf` (um array NumPy de 3 números). Compare com a fórmula `(2 / n) · Xᵀ (X @ w − y)`, guardada em `grad_formula`.

b) Escreva `passo(taxa)`: calcula o gradiente com a fita e atualiza `w_q3` **no lugar** (`w_q3.assign_sub(taxa * gradiente)`), devolvendo o valor da perda como `float`. Depois execute 300 passos com taxa 0,1.

In [ ]:
rng = np.random.default_rng(1)
X_q3 = tf.constant(rng.normal(size=(200, 3)), dtype=tf.float32)
y_q3 = tf.constant(X_q3.numpy() @ np.array([1.5, -2.0, 0.5], dtype="float32") + 0.1 * rng.normal(size=200).astype("float32"))
w_q3 = tf.Variable([0.0, 0.0, 0.0])

In [ ]:
# ✏️ Questão 3
# SEU CÓDIGO AQUI
grad_tf = ...
grad_formula = ...


def passo(taxa):
    # SEU CÓDIGO AQUI
    ...


perdas_q3 = [passo(0.1) for _ in range(300)]
print("pesos encontrados:", w_q3.numpy().round(3), "· perda final:", round(perdas_q3[-1], 4))

In [ ]:
# ✅ Conferência da questão 3 (não altere)
assert np.asarray(grad_tf).shape == (3,) and np.allclose(grad_tf, grad_formula, atol=1e-4), "o gradiente da fita e o da fórmula não coincidem"
assert np.abs(grad_tf).sum() > 0.1
_referencia = np.linalg.lstsq(X_q3.numpy(), y_q3.numpy(), rcond=None)[0]
assert np.allclose(w_q3.numpy(), _referencia, atol=1e-2), f"os pesos ({w_q3.numpy().round(3)}) deveriam chegar perto da solução de mínimos quadrados ({_referencia.round(3)})"
assert perdas_q3[-1] < perdas_q3[0] and perdas_q3[-1] < 0.02
print("Formato correto: a descida do gradiente em TensorFlow chegou à solução de mínimos quadrados.")

**3c)** Escreva, lado a lado, as linhas do seu `passo` e as linhas equivalentes em PyTorch (aula 07, seção 4). Em qual dos dois é preciso zerar os gradientes, e por quê? O que aconteceria se você trocasse `w_q3.assign_sub(...)` por `w_q3 = w_q3 - taxa * gradiente`?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 4 (2,0 pontos): modernizar um código antigo

O trecho abaixo, de 2020, treina um classificador binário em dados tabulares. Ele usa o estilo `tf.keras` da época e contém um erro silencioso.

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

model = Sequential()
model.add(Dense(16, activation="relu", input_dim=8))
model.add(Dense(1, activation="sigmoid"))
model.compile(optimizer=Adam(lr=0.01), loss=BinaryCrossentropy(from_logits=True), metrics=["accuracy"])
model.fit_generator(gerador, epochs=20)
model.save("classificador.h5")
```

Reescreva-o em Keras 3, corrigindo o erro: crie `modelo_q4`, compile, treine por 20 épocas com os arrays `X_q4` e `y_q4` abaixo (lotes de 32, `verbose=0`), guarde o resultado do `fit` em `historico_q4` e salve o modelo no formato atual, com o nome `classificador.keras`.

In [ ]:
rng = np.random.default_rng(3)
X_q4 = rng.normal(size=(2000, 8)).astype("float32")
y_q4 = (X_q4[:, 0] - 0.5 * X_q4[:, 1] + 0.3 * X_q4[:, 2] ** 2 > 0.2).astype("float32")

In [ ]:
# ✏️ Questão 4
keras.utils.set_random_seed(SEMENTE)
# SEU CÓDIGO AQUI
modelo_q4 = ...
historico_q4 = ...

In [ ]:
# ✅ Conferência da questão 4 (não altere)
_ativacao = modelo_q4.layers[-1].activation.__name__
_de_logits = bool(getattr(modelo_q4.loss, "from_logits", False))
assert (_ativacao == "linear") == _de_logits, f"a última camada usa '{_ativacao}' e a perda está com from_logits={_de_logits}: as duas convenções estão misturadas"
assert modelo_q4.count_params() == 8 * 16 + 16 + 16 + 1 and len(historico_q4.history["loss"]) == 20
assert historico_q4.history["accuracy"][-1] > 0.9, "a acurácia ficou baixa"
assert Path("classificador.keras").exists()
print(f"Formato correto: última camada '{_ativacao}', from_logits={_de_logits}, acurácia {historico_q4.history['accuracy'][-1]:.1%}.")

**4b)** Liste todas as alterações que você fez, separando as que eram **necessárias para o código rodar** das que corrigem o **erro silencioso**. Explique o erro: o que a combinação original fazia com as saídas da rede, e por que o treino não acusava nada?

✏️ **Sua resposta:**

_Escreva aqui._

## Questão 5 (1,0 ponto): que framework usar

a) Para cada situação, diga o que você recomendaria e por quê, em duas frases:

1. Uma equipe vai começar um projeto de classificação de imagens médicas, partindo de modelos pré-treinados.
2. Uma empresa tem um sistema de recomendação em TensorFlow 2, em produção há quatro anos, funcionando bem.
3. Você encontrou no GitHub o código de um artigo de 2018, cheio de `tf.Session` e `tf.placeholder`, e precisa reproduzir o resultado.

b) Declare como usou assistentes de IA nesta lista: em quais questões, para quê, e o que precisou corrigir.

✏️ **Sua resposta:**

_Escreva aqui._